# Workshop Lab 3: Pandas Data Cleaning Basics and Script Tools
Welcome! In this notebook, we are going to take raw, messy wildlife observation data (from iNaturalist, on Greater Roadrunners in the Southwest) and clean it using Pandas. We will also look at making script tools inside ArcPro.

Let's start by loading in the csv we exported at the end of notebook 2.

In [ ]:
# Import the tools we need
import pandas as pd

# Load our small sample dataset
# (Notice the relative path: we go up one folder from where this notebook lives, then into data, then working)
file_path = "../04_data/working/smroadrunners_sorted_bydate.csv"
df = pd.read_csv(file_path)

# Quick check: How big is our dataset?
print(f"Total rows and columns: {df.shape}")

# Look at the first 3 rows
df.head(3)

Let's look at the time and date columns. There are four! Even in the first three rows, we can see variation in format, data included, and missing values. This data set needs some cleaning.

### Step 1: Wrangling Time

Let's start by getting a better handle on the time data. Since we've already loaded our csv and imported pandas, we can skip those lines in the chunk below. We are going to add a new column called `date_dt` (dt for date-time), which converts text strings to actual dates. Then we will resort the data using that column.

In [ ]:
# Convert 'observed_on' from text to a true Pandas datetime (dt) object
df['date_dt'] = pd.to_datetime(df['observed_on'], errors='coerce')
# Note: This adds a new column called date_dt - in general it is good practice to avoid overwriting original data

# Now let's test sorting chronologically (oldest to newest) using the real datetime column!
df = df.sort_values(by='date_dt', ascending=True)

print("--- Properly Sorted Chronological Observations ---")
display(df[['observed_on', 'common_name']].head(3))

Question: What is the earliest observation in the dataset?

Challenge: How would you flip this sort to find the most recent observation in the dataset? (Suggestion: use df_newest so you don't overwrite the stored df variable that is sorted oldest to newest).

<details>
<summary><b>Click here for code</b></summary>

```
# Convert 'observed_on' from text to a true Pandas datetime (dt) object
df['date_dt'] = pd.to_datetime(df['observed_on'], errors='coerce')
# Note: This adds a new column called date_dt - in general it is good practice to avoid overwriting original data

# Now let's test sorting chronologically using the real datetime column!
df_newest = df.sort_values(by='date_dt', ascending=False)

print("--- Re-Sorted Chronological Observations ---")
display(df_newest[['observed_on', 'common_name']].head(3))
```
</details>


### Step 1b: Adding Another Calculated Column
In GIS work, we often want to categorize data or flag records based on conditions (e.g., splitting pre-2015 data from recent observations). Let's extract the year from our new datetime column to create a brand new `obs_year` column.

In [ ]:
# Create a new column called obs_year containing just the year of the observation
df['obs_year'] = df['date_dt'].dt.year

# Preview the new column
display(df[['observed_on', 'obs_year', 'common_name']].head(3))

Run the code chunk below to check how many rows and columns we have now:

In [ ]:
# Returns a tuple: (number of rows, number of columns)
print(df.shape)

# Or you can unpack them into separate variables:
num_rows, num_cols = df.shape
print(f"Rows: {num_rows}, Columns: {num_cols}")

We should be at 42 columns but still 200 rows.

### Step 1c: Keeping Only What We Need
Right now, our dataset has 42 columns. Most of those (like URLs, user IDs, and other metadata) are noise that we don't need for our wildlife analysis. And since this is data generated by the public, many of those columns are empty! Let's get rid of them by creating a new dataset of the columns we actually care about and dropping the rest.

In [ ]:
# First let's check our column names again: (run this)
df.head()


In [ ]:
# Define the exact columns we want to keep - note that the order we use is the order the columns will be in in the new data frame (see date_dt at front of list even though the original column is second to last)
columns_to_keep = [
    'id', 
    'date_dt', 
    'time_zone', 
    'latitude', 
    'longitude', 
    'scientific_name', 
    'common_name',
    'obs_year'
]

# Overwrite df to only keep those columns; this does NOT change the original csv in our working folder, just the df variable stored in this notebook.
df = df[columns_to_keep].copy()

# Check our new shape
print(f"New DataFrame shape: {df.shape}")
df.head(3)

Now we only have 8 columns! And our snippet fits nicely in our notebook.

### Step 2: Missing and Messy Data
Real-world data is messy. Let's look for missing coordinates (`latitude` or `longitude`) or weird string variations in the species names.

In [ ]:
# Check for missing values in key columns
print("Missing latitudes:", df['latitude'].isnull().sum())
print("Missing longitudes:", df['longitude'].isnull().sum())

# Let's see what unique common names are hiding in this dataset
print("Unique common names:", df['common_name'].unique())

Thankfully none of these points have missing locations! But we do see that there are multiple spellings for roadrunners.

### Step 3: Cleaning the Data
We want to:
1. Drop any rows missing coordinates (since we can't map them!) - this is not applicable to this csv, but I've included the code below as an example.
2. Standardize the common name typos (fixing 'Roadrunners' to 'Greater Roadrunner').

In [ ]:
# 1. Drop rows with missing latitude or longitude
df_clean = df.dropna(subset=['latitude', 'longitude']).copy()

# 2. Fix the name variations using title casing and replace
df_clean['common_name'] = df_clean['common_name'].str.title().replace({
    'Roadrunners': 'Greater Roadrunner'
})

# Verify our cleanup worked
print(f"Rows before cleaning: {len(df)}")
print(f"Rows after cleaning: {len(df_clean)}")
print("Cleaned unique names:", df_clean['common_name'].unique())

This particular dataset does not have any missing locations, so the number of rows has not changed.

### Step 4: Saving Our Work
Now that our data is clean, let's export it so we can bring it into ArcGIS Pro!

In [ ]:
import os

# Ensure the clean data folder exists and create if it doesn't
os.makedirs("../04_data/clean", exist_ok=True)

# Export to CSV
output_path = "../04_data/clean/smroadrunners_cleaned.csv"
df_clean.to_csv(output_path, index=False)

print(f"Success! Cleaned data saved to: {output_path}")

## Scripts
One of the most valuable aspects of coding is replication. We can take a data workflow, such as this cleaning script, and pull it into ArcPro as a script tool to use with other data. Let's do that now!

Here is what the entire cleaning workflow looks like combined into a single, cohesive Python script. You can inspect how all the pieces we just built (loading, subsetting, cleaning, sorting, and exporting) fit together:

In [ ]:
# CLEANING SCRIPT

import os
import pandas as pd

print("--- Starting Data Cleaning Script ---")

# 1. Define file paths (relative to the script location)
input_path = "../04_data/raw/01_smroadrunners_data.csv" # our messy raw data
output_path = "../04_data/clean/smroadrunners_cleaned_script.csv" # our nice and clean data

# 2. Load data into a df
df = pd.read_csv(input_path)
print(f"Loaded raw data with shape: {df.shape}")

# 3. Convert dates and create calculated year column
df['date_dt'] = pd.to_datetime(df['observed_on'], errors='coerce')
df['obs_year'] = df['date_dt'].dt.year

# 4. Filter down to the columns we actually want
columns_to_keep = [
    'id', 
    'date_dt', 
    'time_zone', 
    'latitude', 
    'longitude', 
    'scientific_name', 
    'common_name',
    'obs_year'
]
df = df[columns_to_keep].copy()

# 5. Clean data (drop missing coordinates, fix text casing/typos)
df_clean = df.dropna(subset=['latitude', 'longitude']).copy()
df_clean['common_name'] = df_clean['common_name'].str.title().replace({
    'Roadrunners': 'Greater Roadrunner'
})

# 6. Sort chronologically
df_clean = df_clean.sort_values(by='date_dt', ascending=True)

# 7. Ensure output directory exists and export
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_clean.to_csv(output_path, index=False)

print(f"Success! Cleaned data saved to: {output_path}")
print(f"Final row count: {len(df_clean)}")

Go ahead and open up both csv files in the 'clean' data folder. Pick a row at random (e.g. row 95) and compare it between the two spreadsheets. They should be identical! This single cell can be saved as a .py (python) file and run in any python environment with pandas installed. Let's practice this now!
- Go back up to the Cleaning Script cell, click into it (edit mode), highlight all the code (ctrl+A), copy it (ctrl+C). 
- Then in the VS Code file explorer, find the folder called `05_scripts` and right click on it. Select `New File` and type in `CleaningScript.py`. **NOTE** When you hit enter VS Code will open a new tab! You can navigate back to this one by clicking on the `03_data_wrangling.ipynb` tab. 
- Paste your copied code into the `CleaningScript.py` tab. Save it (ctrl+S). 
- This is a standalone python script. Because this workshop is focused on ArcPro, we will not run this here (you are welcome to try, but you will likely get an error because of the way arcgispro-py3 handles file paths).

Let's see the power of scripts in action! We've been working with `01_smroadrunners_data.csv`, which is a 200 row subset of the entire roadrunners dataset. Let's take a quick look at `02_roadrunners_data.csv`.

In [ ]:
# Import the tools we need
import pandas as pd

# Load our small sample dataset
# (Notice the relative path: we go up one folder, into data, then raw)
file_path = "../04_data/raw/02_roadrunners_data.csv"
df = pd.read_csv(file_path)

# Quick check: How big is our dataset?
print(f"Total rows and columns: {df.shape}")

# Look at the first 3 rows
df.head(3)

This is a much bigger dataset with over 10,000 observations! Now let's put our script to work. We are going to rerun the cleaning script but point it at our full dataset.
- the input path should point at `02_roadrunners_data.csv` and 
- the output file should be `all_roadrunners_cleaned.csv`

Run the script in the cell below:

In [ ]:
# CLEANING SCRIPT

import os
import pandas as pd

print("--- Starting Data Cleaning Script ---")

# 1. Define file paths (relative to the script location)
input_path = "../04_data/raw/02_roadrunners_data.csv"
output_path = "../04_data/clean/all_roadrunners_cleaned.csv"

# 2. Load data
df = pd.read_csv(input_path)
print(f"Loaded raw data with shape: {df.shape}")

# 3. Convert dates and create calculated year column
df['date_dt'] = pd.to_datetime(df['observed_on'], errors='coerce')
df['obs_year'] = df['date_dt'].dt.year

# 4. Filter down to the columns we actually want
columns_to_keep = [
    'id', 
    'date_dt', 
    'time_zone', 
    'latitude', 
    'longitude', 
    'scientific_name', 
    'common_name',
    'obs_year'
]
df = df[columns_to_keep].copy()

# 5. Clean data (drop missing coordinates, fix text casing/typos)
df_clean = df.dropna(subset=['latitude', 'longitude']).copy()
df_clean['common_name'] = df_clean['common_name'].str.title().replace({
    'Roadrunners': 'Greater Roadrunner'
})

# 6. Sort chronologically
df_clean = df_clean.sort_values(by='date_dt', ascending=True)

# 7. Ensure output directory exists and export
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_clean.to_csv(output_path, index=False)

print(f"Success! Cleaned data saved to: {output_path}")
print(f"Final row count: {len(df_clean)}")

The output tells you how long this took - less than a second! 

## Turning Our Script Into an ArcGIS Pro Tool (`.atbx`)
Now that we have a working standalone script, how do we give it a graphical user interface (GUI) so other GIS analysts (or non-coders) can use it in ArcGIS Pro? ESRI provides us with an easy path forward through script tools and arc toolboxes.

Earlier you copied the CleaningScript into a .py file. Go ahead and open the clean_inaturalist_data.py file now (in 05_scripts folder). This script does the same job as our CleaningScript.py but there are some differences. Take a look at the code chunk below (copied from clean_inaturalist_data.py):



In [ ]:
import os
import pandas as pd
import arcpy

arcpy.AddMessage("Starting Data Cleaning Tool...")

# Get parameters from the ArcGIS Pro Tool dialog box
input_csv = arcpy.GetParameterAsText(0)
output_folder = arcpy.GetParameterAsText(1)
output_filename = arcpy.GetParameterAsText(2)

The arcpy parameters are what allow us to use toolboxes as a GUI. Let's make our script tool! 

Note: If you RUN the chunk above, nothing outputs. This is normal.

### Steps to Build Your Script Tool:
1. **Open ArcGIS Pro** and open IntroToPythonAndArcade.aprx.
2. In the **Catalog pane**, under Toolbox, right click, create a new Toolbox and call it `PythonIsCool.atbx`.
3. Right click on your new toolbox, then **New > Script Tool**.
4. **General Tab:** Name your tool (e.g., `CleanData`) and give it a label (e.g. `Clean Data`).
5. **Parameters Tab:** Add your inputs and outputs:
   * **Parameter 1:** Label it `Input CSV`, Data Type = `File`
   * **Parameter 2:** Label it `Output folder`, Data Type = `Folder`
   * **Parameter 3:** Label it `Output file`, Data Type = `String` (set direction to *Output*)
6. **Execution Tab:** Point the tool to the `clean_inaturalist_data.py` script file in folder 05_scripts.
7. Click **Finish**! You can now double-click your tool in ArcGIS Pro and run it just like any native Esri tool.
8. Make sure to put .csv after your filename in output!

**If you have trouble adding the script tool, use the one in IntroToPythonAndArcade.atbx just to see how it works.**

#### Linked vs. embedded scripts
A script tool can use your Python code in two ways. The **Import script** checkbox on the Execution tab chooses between them.

- **Linked (File path for script is shown):** The tool remembers where the `.py` file lives and reads it each time you run the tool. If you edit and save the `.py` file, the next run uses your changes. The downside is that if the `.py` file is moved, renamed, or deleted, the tool breaks.
- **Embedded (Click on toolbox icon next to file path - should change to *Embedded*):** Pro copies your code *inside* the toolbox file (`.atbx`). The tool no longer needs the `.py` file, so you can move or share the toolbox on its own. The downside is that later edits to the `.py` file don't change the tool, because the tool is now using its own copy.
- You *can* edit scripts in the script tool properties popup! But this is not a good place to test code.

A good rule of thumb is to link while you're building and testing, and embed when you hand the tool to someone else.

Let's test the tool out! Use 03_coyotes_data.csv as your input. Specify the 'clean' folder as your output folder and call the output file cleaned_coyotes_data.csv.

Open up the cleaned csv file. Answer the following questions:
- What was the earliest year of observation for this dataset?
- How many observations are there total?
- How many columns are there?

### Second Script Tool: XYtoPoint.py
Use what you learned above to add a second script tool (if you have trouble you can also find this one in the IntroToPythonAndArcade.atbx).

Parameters:
   * **Parameter 1:** Label it `Input CSV`, Data Type = `File`
   * **Parameter 2:** Label it `Output feature class`, Data Type = `Feature Class`

1) Run the script using the `all_roadrunners_cleaned.csv` with `all_roadrunners_cleaned_XYtoPoint` (default) as the output name. 
2) Now run the script again on the `cleaned_coyotes_data.csv` with `cleaned_coyotes_data_XYtoPoint` (default) as the output name.
- Outputs from this tool go to the project geodatabase by default.


### Next step: Open 03b_historytopython.ipynb